In [6]:
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
# Opcjonalne usunięcie ostrzeżeń (oprócz zmian w kodzie):
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

try:
    plt.style.use('seaborn-v0_8-white')
except:
    plt.style.use('default') 


# --- 1. Odczytywanie i przetwarzanie danych (Wklejony przykładowy tekst) ---
data = """
Artificial intelligence (AI) is intelligence—perceiving, synthesizing, and inferring information—demonstrated by machines, as opposed to intelligence displayed by animals and humans. Example of AI are chatbots, autonomous vehicles, and deep neural networks. Deep learning has dominated the field, and this technique has proved highly successful, helping to solve many challenging problems throughout industry and academia. The various sub-fields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include reasoning, knowledge representation, planning, learning, natural language processing, perception, and the ability to move and manipulate objects. General intelligence (the ability to solve an arbitrary problem) is among the field's long-term goals. Computer scientists and philosophers have since suggested that AI may become an existential risk to humanity if its rational capacities are not steered towards beneficial goals.
"""

chars = list(set(data))
data_size, X_size = len(data), len(chars)
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }


# --- 2. Stałe i hiperparametry (Wariant 10: H_size=50, T_steps=50) ---
H_size = 50   # Rozmiar warstwy ukrytej (Wariant 10)
T_steps = 50  # Długość sekwencji (Wariant 10)

learning_rate = 1e-1
weight_sd = 0.1
z_size = H_size + X_size


# --- 3. Funkcje pomocnicze ---
def sigmoid(x): 
    return 1 / (1 + np.exp(-x))

def dsigmoid(y): 
    return y * (1 - y)

def dtanh(x): 
    return 1 - x * x


# --- 4. Inicjalizacja wag ---
Wf = np.random.randn(H_size, z_size) * weight_sd + 0.5
Wi = np.random.randn(H_size, z_size) * weight_sd + 0.5
Wc = np.random.randn(H_size, z_size) * weight_sd
Wo = np.random.randn(H_size, z_size) * weight_sd + 0.5
Wv = np.random.randn(X_size, H_size) * weight_sd
bf = np.zeros((H_size, 1))
bi = np.zeros((H_size, 1))
bc = np.zeros((H_size, 1))
bo = np.zeros((H_size, 1))
bv = np.zeros((X_size, 1))

parameters = {'Wf': Wf, 'Wi': Wi, 'Wc': Wc, 'Wo': Wo, 'Wv': Wv, 
              'bf': bf, 'bi': bi, 'bc': bc, 'bo': bo, 'bv': bv}

smooth_grads = {param: np.zeros_like(parameters[param]) for param in parameters}


# --- 5. Funkcja Forward Pass (Krok do przodu) ---
def forward_pass(inputs, targets, h_prev, c_prev):
    xs, hs, cs, zs, os, ys, ps, fs, is_, cs_bar, os_gate = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
    hs[-1] = np.copy(h_prev)
    cs[-1] = np.copy(c_prev)
    loss = 0
    
    for t in range(len(inputs)):
        xs[t] = np.zeros((X_size, 1))
        xs[t][inputs[t]] = 1
        
        # Zmieniono np.row_stack na np.vstack, aby usunąć ostrzeżenie
        zs[t] = np.vstack((hs[t-1], xs[t]))

        fs[t] = sigmoid(np.dot(Wf, zs[t]) + bf)
        is_[t] = sigmoid(np.dot(Wi, zs[t]) + bi)
        cs_bar[t] = np.tanh(np.dot(Wc, zs[t]) + bc)
        os_gate[t] = sigmoid(np.dot(Wo, zs[t]) + bo)

        cs[t] = fs[t] * cs[-1] + is_[t] * cs_bar[t]
        hs[t] = os_gate[t] * np.tanh(cs[t])

        os[t] = np.dot(Wv, hs[t]) + bv
        ys[t] = np.exp(os[t]) / np.sum(np.exp(os[t]))
        loss += -np.log(ys[t][targets[t], 0])

    return loss, fs, is_, cs_bar, os_gate, cs, hs, xs, ys


# --- 6. Funkcja Backward Pass (Krok wstecz) ---
def backward_pass(targets, fs, is_, cs_bar, os_gate, cs, hs, xs, ys):
    dWf, dWi, dWc, dWo, dWv = np.zeros_like(Wf), np.zeros_like(Wi), np.zeros_like(Wc), np.zeros_like(Wo), np.zeros_like(Wv)
    dbf, dbi, dbc, dbo, dbv = np.zeros_like(bf), np.zeros_like(bi), np.zeros_like(bc), np.zeros_like(bo), np.zeros_like(bv)
    
    dh_next = np.zeros((H_size, 1))
    dc_next = np.zeros((H_size, 1))
    
    grads = {'Wf': dWf, 'Wi': dWi, 'Wc': dWc, 'Wo': dWo, 'Wv': dWv, 
             'bf': dbf, 'bi': dbi, 'bc': dbc, 'bo': dbo, 'bv': dbv}

    for t in reversed(range(len(targets))):
        dy = np.copy(ys[t])
        dy[targets[t]] -= 1

        dWv += np.dot(dy, hs[t].T)
        dbv += dy
        
        dh = np.dot(Wv.T, dy) + dh_next

        dos_gate = dh * np.tanh(cs[t])
        dos_gate = dsigmoid(os_gate[t]) * dos_gate

        dc = dh * os_gate[t] * dtanh(np.tanh(cs[t])) + dc_next

        dfs = dc * cs[t-1]
        dfs = dsigmoid(fs[t]) * dfs

        dis = dc * cs_bar[t]
        dis = dsigmoid(is_[t]) * dis

        dcs_bar = dc * is_[t]
        dcs_bar = dtanh(cs_bar[t]) * dcs_bar
        
        # Zmieniono np.row_stack na np.vstack, aby usunąć ostrzeżenie
        zs = np.vstack((hs[t-1], xs[t]))

        dWf += np.dot(dfs, zs.T)
        dbf += dfs
        dWi += np.dot(dis, zs.T)
        dbi += dis
        dWc += np.dot(dcs_bar, zs.T)
        dbc += dcs_bar
        dWo += np.dot(dos_gate, zs.T)
        dbo += dos_gate

        dh_next = np.dot(Wf[:, :H_size].T, dfs) + \
                  np.dot(Wi[:, :H_size].T, dis) + \
                  np.dot(Wc[:, :H_size].T, dcs_bar) + \
                  np.dot(Wo[:, :H_size].T, dos_gate)
        
        dc_next = dc * fs[t]

    for dparam in grads.values():
        np.clip(dparam, -5, 5, out=dparam)

    return grads, hs[-1], cs[-1]


# --- 7. Funkcja aktualizacji wag (AdaGrad) ---
def update_params(grads, params, smooth_grads):
    for param in grads:
        g = grads[param]
        smooth_grads[param] += g * g
        params[param] -= learning_rate * g / (np.sqrt(smooth_grads[param]) + 1e-8)


# --- 8. Funkcja samplująca (generowanie tekstu) ---
def sample(h_prev, c_prev, seed_ix, n):
    x = np.zeros((X_size, 1)) 
    x[seed_ix] = 1
    ixes = []
    
    h = h_prev
    c = c_prev
    
    for t in range(n):
        # Zmieniono np.row_stack na np.vstack, aby usunąć ostrzeżenie
        z = np.vstack((h, x))
        
        f = sigmoid(np.dot(Wf, z) + bf)
        i = sigmoid(np.dot(Wi, z) + bi)
        c_bar = np.tanh(np.dot(Wc, z) + bc)
        o = sigmoid(np.dot(Wo, z) + bo)
        
        c = f * c + i * c_bar
        h = o * np.tanh(c)

        out = np.dot(Wv, h) + bv
        p = np.exp(out) / np.sum(np.exp(out))
        
        ix = np.random.choice(range(X_size), p=p.ravel())
        
        x = np.zeros((X_size, 1))
        x[ix] = 1
        ixes.append(ix)
        
    return ixes, h, c


# --- 9. Główna pętla treningowa ---
def training_loop():
    global Wf, Wi, Wc, Wo, Wv, bf, bi, bc, bo, bv
    
    h_prev = np.zeros((H_size, 1))
    c_prev = np.zeros((H_size, 1))
    
    n, p = 0, 0
    moving_avg_loss = -np.log(1.0/X_size) * T_steps
    
    while n <= 10000:
        if p + T_steps + 1 >= data_size or n == 0:
            h_prev = np.zeros((H_size, 1))
            c_prev = np.zeros((H_size, 1))
            p = 0
            
        inputs = [char_to_ix[ch] for ch in data[p:p + T_steps]]
        targets = [char_to_ix[ch] for ch in data[p + 1:p + T_steps + 1]]
        
        loss, fs, is_, cs_bar, os_gate, cs, hs, xs, ys = forward_pass(inputs, targets, h_prev, c_prev)
        
        moving_avg_loss = moving_avg_loss * 0.999 + loss * 0.001
        
        grads, h_prev, c_prev = backward_pass(targets, fs, is_, cs_bar, os_gate, cs, hs, xs, ys)
        
        update_params(grads, parameters, smooth_grads)

        if n % 500 == 0:
            seed_ix = inputs[0] if inputs else char_to_ix[chars[0]]
            sample_ix, _, _ = sample(h_prev, c_prev, seed_ix, 200)
            txt = ''.join(ix_to_char[ix] for ix in sample_ix)
            
            print(f'Iteracja: {n}, Strata: {moving_avg_loss:8.4f}')
            print('--- GENEROWANY TEKST ---')
            print(txt)
            print('------------------------')
            
        p += T_steps
        n += 1

    print(f'Trening zakończony po {n-1} iteracjach. Ostateczna strata: {moving_avg_loss:8.4f}')


# --- Uruchomienie pętli treningowej (GŁÓWNY KROK) ---
print("Rozpoczynanie treningu dla Wariantu 10 (H_size=50, T_steps=50, dane: Wklejony tekst)...")
training_loop()

Rozpoczynanie treningu dla Wariantu 10 (H_size=50, T_steps=50, dane: Wklejony tekst)...
Iteracja: 0, Strata: 186.8831
--- GENEROWANY TEKST ---
avceb cpcriq.iCeir—relApraA'AsltwlnciepI) AppcnGeccsmocnrrcplrr—ltlAltlADe(kq ltpp)l)lsnllcaIAlnyr Alh)enap nnwrresfxAq)Al—  lporr tlllmnAleccaApelqeawDlreAdlcGcy)lAovvyiA))ilArtveleespplncAp fl.rcrnl
------------------------
Iteracja: 500, Strata: 170.7690
--- GENEROWANY TEKST ---
 xgdiirnlepulosyieled nhs nvsesriliaaesliiine—aensfltAbbgeisd gs nil inllsseemtails naecanseiedaw tlnyaaiel ptgtnoai,paatealsddaiefeesrsciaelmlln sarancmleaelocnavseaulinhoerbs taseaGesifseifnilno se 
------------------------
Iteracja: 1000, Strata: 159.3891
--- GENEROWANY TEKST ---
Trrwbuaerrnnresol illtnlsst sothlaliieo.nuntrtrs,e,yuenn n  in sn ca  aega ae rygoarapntg neat cC hem idoedaaol ln rgo spibtda anonrapggmaaoggdden lacgh o,neegiiccn ctdha endat baeaegotca(seratesnt fe
------------------------
Iteracja: 1500, Strata: 152.4945
--- GENEROWANY TEKST ---
 sAele